<a href="https://colab.research.google.com/github/Nitin-ops320/cricanalytics-ai/blob/main/cric_analytics_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Force install an older, stable version of MediaPipe that includes mp.solutions
!pip install opencv-python mediapipe==0.10.21 ultralytics openpyxl
print("✅ Libraries updated with stable MediaPipe version!")

✅ Libraries updated with stable MediaPipe version!


In [ ]:
import cv2

# PASTE YOUR COPIED VIDEO PATH HERE
video_path = '/content/VID_20260602233712.mp4'

# Open the video file
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("❌ Error: Could not open the video file. Please check the path.")
else:
    # Get basic video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps

    print(f"✅ Video loaded successfully!")
    print(f"🎥 Resolution: {width}x{height}")
    print(f"⚡ Frame Rate: {fps:.2f} FPS")
    print(f"🎞️ Total Frames: {frame_count}")
    print(f"⏱️ Video Duration: {duration:.2f} seconds")

    # Read the first frame just to test extraction
    ret, frame = cap.read()
    if ret:
        print(f"📸 Successfully extracted the first frame. Shape: {frame.shape}")

    # Always release the video object when done
    cap.release()

✅ Video loaded successfully!
🎥 Resolution: 480x864
⚡ Frame Rate: 25.00 FPS
🎞️ Total Frames: 53
⏱️ Video Duration: 2.12 seconds
📸 Successfully extracted the first frame. Shape: (864, 480, 3)


In [ ]:
import cv2
import mediapipe as mp

# 1. Initialize MediaPipe Pose and Drawing Utilities
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,       # 0=fast/less accurate, 1=balanced, 2=slow/highly accurate
    enable_segmentation=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# 2. Paths for input and output videos
input_video_path = '/content/VID_20260602233712.mp4'  # Make sure this matches your uploaded file name!
output_video_path = '/content/cricket_pose_output.mp4'

# Open the input video
cap = cv2.VideoCapture(input_video_path)
if not cap.isOpened():
    print("❌ Error: Could not open input video.")
    exit()

# Get video properties to configure the output file writer
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec for MP4 format

# Initialize VideoWriter to save our annotated video
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

print("Processing video frames... This might take a minute depending on video length.")

frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break  # End of video reached

    frame_count += 1

    # MediaPipe requires RGB images, but OpenCV reads in BGR format
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Process the frame to detect pose landmarks
    results = pose.process(rgb_frame)

    # If landmarks are detected, draw them onto the frame
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(
            frame,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2), # Landmark circles (Green)
            mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=2)  # Connection lines (Red)
        )

    # Write the annotated frame to the output video file
    out.write(frame)

# Clean up resources
cap.release()
out.release()
pose.close()

print(f"✅ Finished processing! Processed {frame_count} frames.")
print(f"📁 Your output video has been saved to: {output_video_path}")

Processing video frames... This might take a minute depending on video length.
✅ Finished processing! Processed 53 frames.
📁 Your output video has been saved to: /content/cricket_pose_output.mp4


In [ ]:
import cv2
import mediapipe as mp
import numpy as np

# 1. Function to calculate the angle between three joints
def calculate_angle(a, b, c):
    a = np.array(a) # First point (e.g., Shoulder)
    b = np.array(b) # Mid point (e.g., Elbow)
    c = np.array(c) # End point (e.g., Wrist)

    ba = a - b
    bc = c - b

    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)

    angle = np.degrees(np.arccos(cosine_angle))
    return int(angle)

# 2. Initialize MediaPipe
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, model_complexity=1)

input_video_path = '/content/VID_20260602233712.mp4' # Match your file name!
cap = cv2.VideoCapture(input_video_path)

print("📐 Analyzing cricket biomechanics frame-by-frame...\n")
print(f"{'Frame':<8} | {'Left Elbow Angle':<18} | {'Left Knee Angle':<16}")
print("-" * 50)

frame_id = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb_frame)

    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark

        # Direct integer indexing to avoid any library naming bugs
        # Left Arm: Shoulder (11) -> Elbow (13) -> Wrist (15)
        left_shoulder = [landmarks[11].x, landmarks[11].y]
        left_elbow    = [landmarks[13].x, landmarks[13].y]
        left_wrist    = [landmarks[15].x, landmarks[15].y]

        # Left Leg: Hip (23) -> Knee (25) -> Ankle (27)
        left_hip      = [landmarks[23].x, landmarks[23].y]
        left_knee     = [landmarks[25].x, landmarks[25].y]
        left_ankle    = [landmarks[27].x, landmarks[27].y]

        # Calculate angles
        elbow_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
        knee_angle = calculate_angle(left_hip, left_knee, left_ankle)

        # Print metrics for every 5th frame to keep it readable
        if frame_id % 5 == 0:
            print(f"{frame_id:<8} | {elbow_angle:<16}° | {knee_angle:<14}°")

cap.release()
pose.close()
print("\n✅ Biomechanical analysis complete!")

📐 Analyzing cricket biomechanics frame-by-frame...

Frame    | Left Elbow Angle   | Left Knee Angle 
--------------------------------------------------
5        | 147             ° | 114           °
10       | 162             ° | 117           °
15       | 73              ° | 110           °
25       | 163             ° | 178           °
35       | 15              ° | 166           °
40       | 27              ° | 164           °
45       | 129             ° | 160           °
50       | 37              ° | 163           °

✅ Biomechanical analysis complete!


In [ ]:
import cv2
import mediapipe as mp
import numpy as np

# 1. Reuse our trusty angle calculation function
def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    ba = a - b
    bc = c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)
    return int(np.degrees(np.arccos(cosine_angle)))

# 2. Initialize MediaPipe Pose and Drawing setup
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
pose = mp_pose.Pose(static_image_mode=False, model_complexity=1)

input_video_path = '/content/VID_20260602233712.mp4'  # Match your filename!
output_video_path = '/content/cricket_telemetry_output.mp4'

cap = cv2.VideoCapture(input_video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

print("🎬 Rendering real-time biomechanical telemetry overlay...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb_frame)

    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark

        # Extract coordinates
        left_shoulder = [landmarks[11].x, landmarks[11].y]
        left_elbow    = [landmarks[13].x, landmarks[13].y]
        left_wrist    = [landmarks[15].x, landmarks[15].y]

        left_hip      = [landmarks[23].x, landmarks[23].y]
        left_knee     = [landmarks[25].x, landmarks[25].y]
        left_ankle    = [landmarks[27].x, landmarks[27].y]

        # Calculate angles
        elbow_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
        knee_angle = calculate_angle(left_hip, left_knee, left_ankle)

        # Draw the standard skeleton wireframe first
        mp_drawing.draw_landmarks(
            frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
            mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=2)
        )

        # Convert normalized coordinates (0 to 1) back to actual video pixels for text placement
        elbow_pixel = (int(left_elbow[0] * width), int(left_elbow[1] * height))
        knee_pixel  = (int(left_knee[0] * width), int(left_knee[1] * height))

        # Render the text boxes onto the frame
        # cv2.putText parameters: (image, text, position, font, scale, color, thickness, line_type)
        cv2.putText(frame, f"{elbow_angle} Deg", (elbow_pixel[0] + 15, elbow_pixel[1]),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)

        cv2.putText(frame, f"{knee_angle} Deg", (knee_pixel[0] + 15, knee_pixel[1]),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2, cv2.LINE_AA)

    out.write(frame)

cap.release()
out.release()
pose.close()

print(f"✅ Success! Video saved to: {output_video_path}")
print("👉 Go to your files panel, hit refresh, and download 'cricket_telemetry_output.mp4'!")

🎬 Rendering real-time biomechanical telemetry overlay...
✅ Success! Video saved to: /content/cricket_telemetry_output.mp4
👉 Go to your files panel, hit refresh, and download 'cricket_telemetry_output.mp4'!


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import json

# 1. Angle calculation function
def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    ba = a - b
    bc = c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)
    return int(np.degrees(np.arccos(cosine_angle)))

# 2. Initialize MediaPipe
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, model_complexity=1)

input_video_path = '/content/VID_20260602233712.mp4' # Match your filename!
cap = cv2.VideoCapture(input_video_path)

# Lists to store angles across the whole video
elbow_angles = []
knee_angles = []

print("📊 Analyzing whole-clip biomechanics and compiling dataset...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb_frame)

    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark

        # Coordinates
        left_shoulder = [landmarks[11].x, landmarks[11].y]
        left_elbow    = [landmarks[13].x, landmarks[13].y]
        left_wrist    = [landmarks[15].x, landmarks[15].y]

        left_hip      = [landmarks[23].x, landmarks[23].y]
        left_knee     = [landmarks[25].x, landmarks[25].y]
        left_ankle    = [landmarks[27].x, landmarks[27].y]

        # Calculate and store angles
        elbow_angles.append(calculate_angle(left_shoulder, left_elbow, left_wrist))
        knee_angles.append(calculate_angle(left_hip, left_knee, left_ankle))

cap.release()
pose.close()

# 3. Compute Summary Statistics
if elbow_angles and knee_angles:
    min_elbow = min(elbow_angles)
    max_elbow = max(elbow_angles)
    min_knee  = min(knee_angles)
    max_knee  = max(knee_angles)

    # 4. Rule-Based Cricket Coaching Logic Engine
    coaching_flags = []

    # Elbow Rule: A straight vertical bat shot shouldn't have an overly collapsed elbow at impact
    if min_elbow < 110:
        coaching_flags.append("Collapsed Front Elbow: The front elbow bent too early, leading to a loss of control and power.")
    elif min_elbow > 155:
        coaching_flags.append("Rigid Arm Swing: The arms are entirely straight, making it harder to adjust to late ball movement.")
    else:
        coaching_flags.append("Good High Elbow: Excellent technical preservation of a high, stable front elbow throughout the execution window.")

    # Knee Rule: Stepping into the shot requires bending the front knee to transfer weight forward
    if min_knee > 145:
        coaching_flags.append("Stiff Front Leg: Insufficient knee flexion detected. The batsman isn't bending into the shot, hurting weight transfer.")
    else:
        coaching_flags.append("Solid Front-Foot Stride: Good weight transfer over the ball with a bent front knee.")

    # 5. Package into Structured JSON
    session_report = {
        "analysis_status": "SUCCESS",
        "video_analyzed": input_video_path.split('/')[-1],
        "metrics_summary": {
            "elbow_angle": {
                "minimum_observed_deg": min_elbow,
                "maximum_observed_deg": max_elbow,
                "range_deg": max_elbow - min_elbow
            },
            "knee_angle": {
                "minimum_observed_deg": min_knee,
                "maximum_observed_deg": max_knee,
                "range_deg": max_knee - min_knee
            }
        },
        "technical_coaching_insights": coaching_flags
    }

    # Convert dictionary to beautifully formatted JSON string
    json_output = json.dumps(session_report, indent=4)
    print("\n🖥️ Generated Structured JSON Payload for LLM Consumption:\n")
    print(json_output)

    # Save JSON file locally to your colab directory
    with open('/content/cricket_metrics.json', 'w') as f:
        f.write(json_output)

else:
    print("❌ Error: No pose landmarks were detected in the video file.")

📊 Analyzing whole-clip biomechanics and compiling dataset...

🖥️ Generated Structured JSON Payload for LLM Consumption:

{
    "analysis_status": "SUCCESS",
    "video_analyzed": "VID_20260602233712.mp4",
    "metrics_summary": {
        "elbow_angle": {
            "minimum_observed_deg": 0,
            "maximum_observed_deg": 178,
            "range_deg": 178
        },
        "knee_angle": {
            "minimum_observed_deg": 106,
            "maximum_observed_deg": 178,
            "range_deg": 72
        }
    },
    "technical_coaching_insights": [
        "Collapsed Front Elbow: The front elbow bent too early, leading to a loss of control and power.",
        "Solid Front-Foot Stride: Good weight transfer over the ball with a bent front knee."
    ]
}


In [ ]:
import json
import google.generativeai as genai
from google.colab import userdata

# 1. Securely fetch the API key from Colab Secrets and configure Gemini
try:
    api_key = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=api_key)
    print("✅ Gemini API successfully configured!")
except Exception as e:
    print("❌ Error: Could not retrieve GEMINI_API_KEY from Colab Secrets.")

# 2. Load the JSON data generated in the previous step
try:
    with open('/content/cricket_metrics.json', 'r') as f:
        metrics_data = json.load(f)
    print("📁 Loaded 'cricket_metrics.json' successfully.")
except FileNotFoundError:
    print("❌ Error: 'cricket_metrics.json' not found.")

# 3. Formulate the professional prompt for the AI Coach
prompt = f"""
You are an elite international cricket batting coach known for deep tactical and biomechanical analysis.
You have been handed a structured JSON report generated by a computer vision system tracking a batsman's performance.

Review the data below carefully:
{json.dumps(metrics_data, indent=2)}

Please write a highly professional, constructive, and actionable coaching report based ONLY on the data provided. Your response must include:
1. An introductory sentence acknowledging the specific shot analyzed.
2. A technical breakdown explaining what the joint angles indicate about their mechanics (praise good elements and explain technical faults).
3. Exactly one highly specific training drill designed to address any technical flags noted in the data.

Keep the tone encouraging, authentic to cricket terminology, and well-structured. Do not output any code or raw markdown headers, just the fluid text of the report.
"""

# 4. Initialize the model using the current stable production endpoint
print("🤖 Connecting to Gemini AI (gemini-3.5-flash) to generate the report...")
try:
    # UPDATED: Changed from 1.5-flash to the current stable 3.5-flash
    model = genai.GenerativeModel('gemini-3.5-flash')
    response = model.generate_content(prompt)

    print("\n🏏 ==================== OFFICIAL AI COACHING REPORT ====================\n")
    print(response.text)
    print("\n========================================================================")

    # Save the text report locally as a file
    with open('/content/coaching_report.txt', 'w') as f:
        f.write(response.text)
    print("\n💾 Report successfully saved to /content/coaching_report.txt")

except Exception as e:
    print(f"❌ Failed to generate report. Error details: {e}")

✅ Gemini API successfully configured!
📁 Loaded 'cricket_metrics.json' successfully.
🤖 Connecting to Gemini AI (gemini-3.5-flash) to generate the report...

🏏 ==================== OFFICIAL AI COACHING REPORT ====================

Following a detailed review of your front-foot drive captured in video session VID_20260602233712.mp4, I have compiled a comprehensive biomechanical analysis to help refine your stroke play.

Looking closely at the telemetry, your lower-body mechanics are exemplary. Your knee angle ranges from a fully extended 178 degrees down to a flexed 106 degrees, representing a highly dynamic 72-degree range of movement. This indicates an exceptionally strong, positive front-foot stride with a bent front knee that allows you to get your head and weight directly over the ball, creating a wonderfully stable base. However, we see a stark contrast in your upper-body mechanics. While your elbow angle shows a wide 178-degree range of motion, the data flags a collapsed front elbo

In [1]:
%%writefile app.py
import streamlit as st
import cv2
import mediapipe as mp
import numpy as np
import json
import google.generativeai as genai
import os

# Safe vector angle calculator (2D Projection)
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return int(np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0))))

# Page Layout configuration
st.set_page_config(page_title="CricAnalytics AI - Pro Engine", layout="wide", initial_sidebar_state="expanded")
st.title("🏏 CricAnalytics AI: Universal Movement Performance Engine")
st.markdown("Advanced computer vision and generative biomechanical analysis scaling across all cricket motions.")

# Sidebar System Configuration
st.sidebar.header("⚙️ Core Configuration")
user_api_key = st.sidebar.text_input("Gemini API Key", type="password")
analysis_mode = st.sidebar.selectbox("Select Analysis Discipline", [
    "Batting Biomechanics & Shot Mechanics",
    "Bowling Action, Release & Stride Dynamics",
    "Fielding, Catching, Throwing & Agility"
])
uploaded_file = st.sidebar.file_uploader("Upload Session Video Clip", type=['mp4', 'mov', 'avi', 'mkv'])

if uploaded_file and user_api_key:
    genai.configure(api_key=user_api_key)

    # Clean staging setup for processing
    input_path = "temp_input.mp4"
    output_path = "telemetry_output.mp4"
    with open(input_path, "wb") as f:
        f.write(uploaded_file.read())

    if st.sidebar.button("🚀 Execute Biomechanical Pipeline"):
        cap = cv2.VideoCapture(input_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # Guard against zero/broken video values
        if total_frames <= 0 or fps <= 0:
            st.error("❌ Invalid or corrupted video file format. Try uploading a different clip.")
            st.stop()

        out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

        # Init Tracking Architecture
        mp_pose = mp.solutions.pose
        pose = mp_pose.Pose(static_image_mode=False, model_complexity=1, min_detection_confidence=0.5, min_tracking_confidence=0.5)
        mp_drawing = mp.solutions.drawing_utils

        # Full-body universal telemetry arrays
        telemetry_data = {
            "left_elbow": [], "right_elbow": [],
            "left_shoulder": [], "right_shoulder": [],
            "left_knee": [], "right_knee": [],
            "left_hip": [], "right_hip": [],
            "center_mass_displacement": [] # Tracking global movement velocity
        }

        progress_bar = st.progress(0)
        status_text = st.empty()

        frame_idx = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            frame_idx += 1
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose.process(rgb_frame)

            if results.pose_landmarks:
                landmarks = results.pose_landmarks.landmark

                # Extract positional mappings
                def get_pt(idx): return [landmarks[idx].x, landmarks[idx].y]

                # Joint extraction mappings (MediaPipe indices)
                p = {
                    "ls": get_pt(11), "le": get_pt(13), "lw": get_pt(15),
                    "rs": get_pt(12), "re": get_pt(14), "rw": get_pt(16),
                    "lh": get_pt(23), "lk": get_pt(25), "la": get_pt(27),
                    "rh": get_pt(24), "rk": get_pt(26), "ra": get_pt(28)
                }

                # Calculate all structural kinematics
                ang = {
                    "le": calculate_angle(p["ls"], p["le"], p["lw"]),
                    "re": calculate_angle(p["rs"], p["re"], p["rw"]),
                    "ls": calculate_angle(p["lh"], p["ls"], p["le"]),
                    "rs": calculate_angle(p["rh"], p["rs"], p["re"]),
                    "lk": calculate_angle(p["lh"], p["lk"], p["la"]),
                    "rk": calculate_angle(p["rh"], p["rk"], p["ra"]),
                    "lh": calculate_angle(p["ls"], p["lh"], p["lk"]),
                    "rh": calculate_angle(p["rs"], p["rh"], p["rk"])
                }

                # Append to datasets
                telemetry_data["left_elbow"].append(ang["le"])
                telemetry_data["right_elbow"].append(ang["re"])
                telemetry_data["left_shoulder"].append(ang["ls"])
                telemetry_data["right_shoulder"].append(ang["rs"])
                telemetry_data["left_knee"].append(ang["lk"])
                telemetry_data["right_knee"].append(ang["rk"])
                telemetry_data["left_hip"].append(ang["lh"])
                telemetry_data["right_hip"].append(ang["rh"])

                # Track position of midpoint hip to compute translational velocity vectors
                mid_hip_x = (p["lh"][0] + p["rh"][0]) / 2.0
                telemetry_data["center_mass_displacement"].append(float(mid_hip_x))

                # Render complete overlay graphics
                mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

                # Draw minimal key contextual telemetry text onto screen graphics
                cv2.putText(frame, f"L_Elbow: {ang['le']}deg", (int(p['le'][0]*width)+10, int(p['le'][1]*height)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1)
                cv2.putText(frame, f"R_Elbow: {ang['re']}deg", (int(p['re'][0]*width)+10, int(p['re'][1]*height)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1)
                cv2.putText(frame, f"L_Knee: {ang['lk']}deg", (int(p['lk'][0]*width)+10, int(p['lk'][1]*height)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0,255,255), 1)
                cv2.putText(frame, f"R_Knee: {ang['rk']}deg", (int(p['rk'][0]*width)+10, int(p['rk'][1]*height)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0,255,255), 1)

            out.write(frame)
            progress_bar.progress(int((frame_idx / total_frames) * 100))
            status_text.text(f"Processing motion telemetry frame {frame_idx}/{total_frames}...")

        cap.release()
        out.release()
        pose.close()

        # Robust evaluation check to completely prevent ValueError on empty datasets
        if not telemetry_data["left_elbow"]:
            status_text.empty()
            progress_bar.empty()
            st.error("❌ **Kinematic Tracking Error:** The software could not reliably isolate human skeletal metrics from this video. Ensure background clarity and clear full-body visibility.")
        else:
            status_text.text("🤖 Constructing Unified Kinematic Payload...")

            # Formulate structured summary dataset metrics mapping max, min, ranges
            def compile_stats(arr):
                return {"min": int(np.min(arr)), "max": int(np.max(arr)), "range": int(np.max(arr) - np.min(arr))}

            packaged_payload = {
                "discipline_context": analysis_mode,
                "kinematic_metrics": {
                    "upper_body_joints": {
                        "left_elbow": compile_stats(telemetry_data["left_elbow"]),
                        "right_elbow": compile_stats(telemetry_data["right_elbow"]),
                        "left_shoulder": compile_stats(telemetry_data["left_shoulder"]),
                        "right_shoulder": compile_stats(telemetry_data["right_shoulder"])
                    },
                    "lower_body_joints": {
                        "left_knee": compile_stats(telemetry_data["left_knee"]),
                        "right_knee": compile_stats(telemetry_data["right_knee"]),
                        "left_hip": compile_stats(telemetry_data["left_hip"]),
                        "right_hip": compile_stats(telemetry_data["right_hip"])
                    }
                },
                "locomotive_dynamics": {
                    "total_frames_processed": frame_idx,
                    "horizontal_displacement_range": float(np.max(telemetry_data["center_mass_displacement"]) - np.min(telemetry_data["center_mass_displacement"]))
                }
            }

            # Orchestrate specialized targeted system prompt logic based on disciplinary track
            system_instruction = f"""
            You are a world-class high-performance cricket sports scientist and national team bio-mechanics coach.
            You are analyzing the kinematic tracking output of a movement sequence categorized under the discipline: "{analysis_mode}".

            Review the raw physical data metrics compiled below:
            {json.dumps(packaged_payload, indent=2)}

            Provide a comprehensive, elite coaching report. Your analysis must adapt to whatever action is in the video based on the data:
            1. If it's batting, interpret what the data indicates about their posture, backlift, stance, or extension.
            2. If it's bowling, evaluate what the joint ranges mean for their loading, alignment, and follow-through.
            3. If it's fielding, interpret what the displacement and flexibility angles suggest about their speed, throwing mechanics, or core agility.

            Structure your report cleanly with sections for: Technical Biomechanical Evaluation, Core Vulnerabilities/Strengths Identified, and 1 Specialized Elite Training Drill.
            Maintain professional, actionable sports terminology. Do not output any markdown code blocks, system strings, or raw script syntax.
            """

            status_text.text("🧠 Deploying Generative Large Language Model Evaluation Layer...")
            try:
                model = genai.GenerativeModel('gemini-3.5-flash')
                response = model.generate_content(system_instruction)

                status_text.empty()
                progress_bar.empty()
                st.success("✅ Analysis Successfully Concluded!")

                col1, col2 = st.columns(2)
                with col1:
                    st.subheader("📊 CV Overlaid Telemetry Output")
                    st.video(output_path)
                with col2:
                    st.subheader("📋 Professional Biomechanical Analysis")
                    st.write(response.text)

            except Exception as e:
                st.error(f"❌ Verification Layer Error: Unable to extract data from Gemini API. Details: {e}")
else:
    st.info("💡 Configuration Status: Open the sidebar, insert your Gemini API Key, select your target cricket discipline, and upload your movement video to initialize processing.")

Writing app.py
